In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 04 · Variables — cada idea es una hipótesis que se mide

> **Esqueleto.** Guía completa: `curso/kit-estudiante/guia-modelado-y-optimizacion.md`, sección 1.

Crear variables es fácil; saber si **sirven** es lo difícil. Cada idea se mide con **el mismo modelo, los
mismos pliegues y la misma métrica**; solo cambia el conjunto de variables. Una idea entra solo si la
mejora supera el **ruido entre pliegues**.

Qué produce este notebook: una **tabla de ideas medidas** con su decisión, y el código reutilizable en
`src/proyecto/features/`. Este notebook solo **orquesta**.

## 0. Cargar los datos de entrenamiento

In [ ]:
import pandas as pd

# === Configuración: completa estas dos líneas (las mismas de tu notebook 03) ===
ID_COL = None     # TODO: columna que identifica a la entidad que se repite (None si no hay entidades repetidas)
TARGET = None     # TODO: tu columna objetivo

train_path = Path.cwd().resolve().parent / "data" / "processed" / "train.joblib"
if not train_path.exists():
    train = None
    print("Todavía no hay datos de entrenamiento: completa y ejecuta antes el notebook 03_preprocesamiento.")
else:
    import joblib
    train = joblib.load(train_path)
    print(f"entrenamiento: {train.shape[0]:,} filas x {train.shape[1]} columnas (la prueba NO se abre hoy)")

listo = train is not None and TARGET is not None
if train is not None and TARGET is None:
    print("Define TARGET (y, si hay entidades repetidas, ID_COL) arriba para continuar.")

## 1. Mis ideas, antes de medirlas

Para cada idea escribe **qué variable es**, **por qué podría ayudar** y **si se conoce al momento de
predecir** (si no, es fuga: descártala).

| # | Idea (variable y cómo se calcula) | Por qué podría ayudar | ¿Se conoce al predecir? |
|---|---|---|---|
| 1 | TODO | TODO | ☐ sí ☐ no |
| 2 | TODO | TODO | ☐ sí ☐ no |
| 3 | TODO | TODO | ☐ sí ☐ no |

**Regla de decisión (escríbela ANTES de medir):** TODO. Ejemplo: *conservo la idea solo si la mejora / ruido
es mayor que 1 y se conoce al predecir.*

## 2. Validación común

Los pliegues se calculan **una vez** y se reutilizan en toda comparación. Si hay entidades repetidas, los
pliegues son **por entidad**.

In [ ]:
if not listo:
    pliegues = None
    print("Sin datos o sin TARGET: esta celda crea los pliegues cuando estén.")
else:
    from sklearn.model_selection import GroupKFold, KFold, StratifiedGroupKFold, StratifiedKFold

    X, y = train.drop(columns=[TARGET]), train[TARGET]
    es_regresion = pd.api.types.is_numeric_dtype(y) and y.nunique() > 20   # ajusta si tu objetivo es otro
    if ID_COL is not None:
        cv = GroupKFold(n_splits=5) if es_regresion else StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
        pliegues = list(cv.split(X, y, groups=X[ID_COL]))
        for i_ent, i_val in pliegues:   # comprobación: ninguna entidad en los dos lados de un pliegue
            assert set(X.iloc[i_ent][ID_COL]).isdisjoint(set(X.iloc[i_val][ID_COL])), "hay entidades en los dos lados"
    else:
        cv = KFold(n_splits=5, shuffle=True, random_state=42) if es_regresion else StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        pliegues = list(cv.split(X, y))
    print(f"{len(pliegues)} pliegues, los MISMOS para todo lo que compares hoy.")

## 3. Pipeline base y métrica

Importa tu función de `src/proyecto/models/train.py` y define la métrica. La **métrica principal** debe
reflejar la decisión del negocio (tu ficha de caso); añade 1–2 **de control**.

In [ ]:
try:
    from proyecto.models.train import build_pipeline   # TODO: crea build_pipeline(modelo, ...) en src/proyecto/models/train.py
except ModuleNotFoundError:
    build_pipeline = None
    print("TODO: crea src/proyecto/models/train.py con build_pipeline(modelo): preprocesamiento + modelo dentro de un Pipeline.")

scoring = None   # TODO: dict de métricas, p. ej. {"costo": make_scorer(tu_costo, greater_is_better=False), "f1": "f1_macro"}
METRICA = None   # TODO: nombre de tu métrica principal dentro de scoring (si es negativa, multiplícala por -1 al leerla)

## 4. El experimento

Un modelo con **parámetros fijos** y una variante por idea. Define en `variantes` qué columnas usa cada una
(la referencia primero).

In [ ]:
modelo_fijo = None   # TODO: un modelo con parámetros fijos (p. ej. LightGBM con sus parámetros de partida)

variantes = {
    # "referencia": {...},   # TODO: las variables actuales, tal como las dejaste en el notebook 03
    # "+ idea 1": {...},     # TODO: lo que cambia respecto de la referencia
}
resultados = {}   # nombre de variante -> DataFrame con una fila por pliegue

faltan = [nombre for nombre, valor in {"pliegues": pliegues, "build_pipeline": build_pipeline, "scoring": scoring,
                                          "modelo_fijo": modelo_fijo, "variantes": variantes}.items()
          if valor is None or (hasattr(valor, "__len__") and len(valor) == 0)]
if faltan:
    print("Pendiente: completa", ", ".join(faltan), "para correr el experimento.")
else:
    from sklearn.base import clone
    from sklearn.model_selection import cross_validate

    for nombre, opciones in variantes.items():
        pipeline = build_pipeline(clone(modelo_fijo), **opciones)   # TODO: adapta cómo `opciones` cambia las variables
        res = cross_validate(pipeline, X, y, cv=pliegues, scoring=scoring, n_jobs=1)
        resultados[nombre] = pd.DataFrame({k.replace("test_", ""): v for k, v in res.items() if k.startswith("test_")})
        print(f"{nombre:30s} listo")

## 5. Mejora sobre ruido

In [ ]:
import numpy as np


def mejora_sobre_ruido(referencia, variante):
    """Valores por pliegue de una métrica donde MENOR es mejor. Positivo: la variante mejora."""
    mejora = np.mean(referencia) - np.mean(variante)
    ruido = np.std(variante, ddof=1)
    return mejora, ruido, (mejora / ruido if ruido > 0 else np.inf)


if not resultados or METRICA is None:
    print("Sin resultados todavía: esta celda construirá tu tabla de mejora / ruido.")
else:
    nombres = list(resultados)
    ref = resultados[nombres[0]][METRICA].abs()   # TODO: quita .abs() si tu métrica ya es "menor es mejor" y positiva
    filas = []
    for nombre in nombres:
        valores = resultados[nombre][METRICA].abs()
        mejora, ruido, razon = mejora_sobre_ruido(ref, valores)
        filas.append({"variante": nombre, "métrica": valores.mean(), "± ruido": ruido, "mejora": mejora, "mejora / ruido": razon})
    tabla = pd.DataFrame(filas).set_index("variante")
    display(tabla.round(4))

## 6. Decisiones

Una fila por idea, **con su cifra** (mejora / ruido). Cópiala a `reports/04_features.md`.

| # | Idea | Mejora / ruido | Decisión | Motivo |
|---|---|---|---|---|
| 1 | TODO | TODO | ☐ conservar ☐ descartar | TODO |
| 2 | TODO | TODO | ☐ conservar ☐ descartar | TODO |

> **Una idea descartada con su cifra es un resultado, no un fracaso.** Anótala igual: documenta lo que ya
> probaste.

## Entregable de esta etapa

- [ ] Cada idea tiene su fila con **mejora / ruido** y una decisión escrita.
- [ ] Todas las variantes se midieron con **los mismos pliegues** (por entidad si hay entidades repetidas).
- [ ] Ninguna variable usa información que no se conoce **al predecir**.
- [ ] El código reutilizable está en `src/proyecto/features/` y el notebook solo lo llama.
- [ ] `reports/04_features.md` completado.